In [3]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm import tqdm
import numpy as np
from datetime import datetime

print("=" * 70)
print("CRYPTOCURRENCY SENTIMENT ANALYSIS (2017-2022)")
print("=" * 70)

# ============================================================
# 1. LOAD AND PREPARE BITCOIN DATA
# ============================================================
print("\n📊 PROCESSING BITCOIN TWEETS")
print("-" * 50)

# Load your existing BTC data (from filtered_tweets_final.csv) - uses semicolon
try:
    btc_original = pd.read_csv('filtered_tweets_final.csv', sep=';')
    btc_original['date'] = pd.to_datetime(btc_original['date'])
    print(f"✓ Original BTC data: {len(btc_original):,} tweets from {btc_original['date'].min().date()} to {btc_original['date'].max().date()}")
except FileNotFoundError:
    print("⚠️ filtered_tweets_final.csv not found - skipping original data")
    btc_original = pd.DataFrame()

# Load new BTC data (2017-2022) - uses semicolon separator
btc_new = pd.read_csv('btc_tweets_2017_2022.csv', sep=';')
btc_new['date'] = pd.to_datetime(btc_new['date'])
print(f"✓ New BTC data: {len(btc_new):,} tweets from {btc_new['date'].min().date()} to {btc_new['date'].max().date()}")
print(f"   Columns: {btc_new.columns.tolist()}")

# Combine BTC datasets
btc_dfs = []
if not btc_original.empty:
    btc_original_renamed = btc_original.rename(columns={'n_replies': 'replies_count'})
    btc_dfs.append(btc_original_renamed[['date', 'text', 'n_retweets', 'n_likes', 'replies_count']])
    
btc_new_renamed = btc_new.rename(columns={'n_replies': 'replies_count'})
btc_dfs.append(btc_new_renamed[['date', 'text', 'n_retweets', 'n_likes', 'replies_count']])

btc_combined = pd.concat(btc_dfs, ignore_index=True)
btc_combined = btc_combined.sort_values('date').reset_index(drop=True)
btc_combined['date_only'] = btc_combined['date'].dt.date

print(f"\n✅ Combined BTC data: {len(btc_combined):,} tweets")
print(f"   Date range: {btc_combined['date'].min().date()} to {btc_combined['date'].max().date()}")

# ============================================================
# 2. LOAD ETHEREUM DATA
# ============================================================
print("\n📊 PROCESSING ETHEREUM TWEETS")
print("-" * 50)

eth_new = pd.read_csv('eth_tweets_2017_2022.csv', sep=';')
eth_new['date'] = pd.to_datetime(eth_new['date'])
eth_new['date_only'] = eth_new['date'].dt.date

print(f"✓ ETH data: {len(eth_new):,} tweets")
print(f"   Columns: {eth_new.columns.tolist()}")
print(f"   Date range: {eth_new['date'].min().date()} to {eth_new['date'].max().date()}")

eth_final = eth_new[['date', 'text', 'n_retweets', 'n_likes', 'n_replies']].rename(
    columns={'n_replies': 'replies_count'})
eth_final['date_only'] = eth_final['date'].dt.date

# ============================================================
# 3. SENTIMENT ANALYSIS FUNCTION
# ============================================================
def analyze_sentiment(df, currency_name):
    """
    Perform sentiment analysis on tweet data
    """
    print(f"\n🔍 Analyzing {currency_name} sentiment for {len(df):,} tweets...")
    
    analyzer = SentimentIntensityAnalyzer()
    
    sentiments = []
    pos_scores = []
    neg_scores = []
    
    for text in tqdm(df['text'], desc=f"Processing {currency_name} tweets"):
        if pd.isna(text) or text == '':
            sentiments.append(0.0)
            pos_scores.append(0.0)
            neg_scores.append(0.0)
        else:
            try:
                scores = analyzer.polarity_scores(str(text))
                sentiments.append(scores['compound'])
                pos_scores.append(scores['pos'])
                neg_scores.append(scores['neg'])
            except:
                sentiments.append(0.0)
                pos_scores.append(0.0)
                neg_scores.append(0.0)
    
    df['sentiment'] = sentiments
    df['positive_score'] = pos_scores
    df['negative_score'] = neg_scores
    
    # Calculate weighted sentiment
    df['engagement_weight'] = df['n_likes'] + df['n_retweets']
    df['weighted_sentiment'] = df['sentiment'] * df['engagement_weight']
    
    return df

# ============================================================
# 4. DAILY AGGREGATION FUNCTION (CSV FRIENDLY OUTPUT)
# ============================================================
def aggregate_daily_sentiment(df, currency_name):
    """
    Aggregate sentiment metrics by day and return CSV-friendly format
    """
    print(f"\n📈 Calculating daily {currency_name} sentiment aggregates...")
    
    # Ensure date_only exists
    if 'date_only' not in df.columns:
        df['date_only'] = df['date'].dt.date
    
    daily_stats = df.groupby('date_only').agg({
        'sentiment': ['mean', 'std', 'count'],
        'positive_score': 'mean',
        'negative_score': 'mean',
        'weighted_sentiment': 'sum',
        'engagement_weight': 'sum',
        'n_likes': 'sum',
        'n_retweets': 'sum'
    }).round(4)
    
    # Flatten columns
    daily_stats.columns = [
        'sentiment_mean', 'sentiment_std', 'tweet_count',
        'positive_mean', 'negative_mean',
        'weighted_sentiment_sum', 'total_engagement_weight',
        'total_likes', 'total_retweets'
    ]
    
    # Calculate weighted average sentiment (avoid division by zero)
    daily_stats['weighted_sentiment_mean'] = daily_stats.apply(
        lambda row: row['weighted_sentiment_sum'] / row['total_engagement_weight'] if row['total_engagement_weight'] > 0 else row['sentiment_mean'],
        axis=1
    ).round(4)
    
    # Calculate sentiment ratio
    daily_stats['pos_neg_ratio'] = (
        daily_stats['positive_mean'] / (daily_stats['negative_mean'] + 0.0001)
    ).round(2)
    
    # Identify sentiment categories
    daily_stats['sentiment_category'] = pd.cut(
        daily_stats['weighted_sentiment_mean'],
        bins=[-1, -0.5, -0.1, 0.1, 0.5, 1],
        labels=['Very Negative', 'Negative', 'Neutral', 'Positive', 'Very Positive']
    )
    
    # Reset index
    daily_sentiment = daily_stats.reset_index()
    daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date_only'])
    
    # Keep exactly the columns in the order you specified
    daily_sentiment = daily_sentiment[[
        'date', 'tweet_count', 'total_likes', 'total_retweets', 'total_engagement_weight',
        'sentiment_mean', 'weighted_sentiment_mean', 'sentiment_std',
        'positive_mean', 'negative_mean', 'pos_neg_ratio', 'sentiment_category'
    ]]
    
    # Format date as YYYY-MM-DD only (no time)
    daily_sentiment['date'] = daily_sentiment['date'].dt.date
    
    return daily_sentiment

# ============================================================
# 5. RUN ANALYSIS FOR BTC
# ============================================================
btc_with_sentiment = analyze_sentiment(btc_combined, "Bitcoin")
btc_daily = aggregate_daily_sentiment(btc_with_sentiment, "Bitcoin")

# Save BTC results with comma separator (CSV friendly)
btc_with_sentiment.to_csv('btc_tweets_with_sentiment_2017_2022.csv', index=False)
btc_daily.to_csv('btc_daily_sentiment_2017_2022.csv', index=False)
print(f"\n💾 Saved BTC results:")
print(f"   • btc_tweets_with_sentiment_2017_2022.csv ({len(btc_with_sentiment):,} tweets)")
print(f"   • btc_daily_sentiment_2017_2022.csv ({len(btc_daily)} days)")

# Show sample of BTC daily file
print("\n📝 BTC Daily Sample (first 5 rows):")
print(btc_daily.head().to_string())

# ============================================================
# 6. RUN ANALYSIS FOR ETH
# ============================================================
eth_with_sentiment = analyze_sentiment(eth_final, "Ethereum")
eth_daily = aggregate_daily_sentiment(eth_with_sentiment, "Ethereum")

# Save ETH results with comma separator (CSV friendly)
eth_with_sentiment.to_csv('eth_tweets_with_sentiment_2017_2022.csv', index=False)
eth_daily.to_csv('eth_daily_sentiment_2017_2022.csv', index=False)
print(f"\n💾 Saved ETH results:")
print(f"   • eth_tweets_with_sentiment_2017_2022.csv ({len(eth_with_sentiment):,} tweets)")
print(f"   • eth_daily_sentiment_2017_2022.csv ({len(eth_daily)} days)")

# Show sample of ETH daily file
print("\n📝 ETH Daily Sample (first 5 rows):")
print(eth_daily.head().to_string())

# ============================================================
# 7. VERIFY THE FORMAT
# ============================================================
print("\n" + "="*70)
print("VERIFYING CSV FORMAT")
print("="*70)

for currency in ['btc', 'eth']:
    filename = f'{currency}_daily_sentiment_2017_2022.csv'
    try:
        df = pd.read_csv(filename)
        print(f"\n📄 {currency.upper()} file: {filename}")
        print(f"   Shape: {df.shape}")
        print(f"   Columns: {df.columns.tolist()}")
        print(f"   First row:")
        print(df.iloc[0].to_string())
    except Exception as e:
        print(f"❌ Error reading {filename}: {e}")

print("\n" + "="*70)
print("✅ ANALYSIS COMPLETE")
print("="*70)

CRYPTOCURRENCY SENTIMENT ANALYSIS (2017-2022)

📊 PROCESSING BITCOIN TWEETS
--------------------------------------------------
✓ Original BTC data: 756,401 tweets from 2019-03-30 to 2022-07-27
✓ New BTC data: 224,967 tweets from 2017-09-21 to 2021-02-28
   Columns: ['date', 'n_replies', 'n_likes', 'n_retweets', 'text']

✅ Combined BTC data: 981,368 tweets
   Date range: 2017-09-21 to 2022-07-27

📊 PROCESSING ETHEREUM TWEETS
--------------------------------------------------
✓ ETH data: 53,805 tweets
   Columns: ['date', 'n_replies', 'n_likes', 'n_retweets', 'text', 'date_only']
   Date range: 2017-09-21 to 2021-02-28

🔍 Analyzing Bitcoin sentiment for 981,368 tweets...


Processing Bitcoin tweets: 100%|█████| 981368/981368 [01:11<00:00, 13680.85it/s]



📈 Calculating daily Bitcoin sentiment aggregates...

💾 Saved BTC results:
   • btc_tweets_with_sentiment_2017_2022.csv (981,368 tweets)
   • btc_daily_sentiment_2017_2022.csv (1258 days)

📝 BTC Daily Sample (first 5 rows):
         date  tweet_count  total_likes  total_retweets  total_engagement_weight  sentiment_mean  weighted_sentiment_mean  sentiment_std  positive_mean  negative_mean  pos_neg_ratio sentiment_category
0  2017-09-21           66      10004.0          6145.0                  16149.0         -0.0112                  -0.0009         0.4802         0.0867         0.0874           0.99            Neutral
1  2017-09-22           93      18332.0         14863.0                  33195.0          0.0630                   0.0222         0.3880         0.0895         0.0592           1.51            Neutral
2  2017-09-23           84      16397.0         10195.0                  26592.0          0.1079                   0.2807         0.3939         0.1168         0.0583       

Processing Ethereum tweets: 100%|██████| 53805/53805 [00:04<00:00, 12501.04it/s]



📈 Calculating daily Ethereum sentiment aggregates...

💾 Saved ETH results:
   • eth_tweets_with_sentiment_2017_2022.csv (53,805 tweets)
   • eth_daily_sentiment_2017_2022.csv (1257 days)

📝 ETH Daily Sample (first 5 rows):
         date  tweet_count  total_likes  total_retweets  total_engagement_weight  sentiment_mean  weighted_sentiment_mean  sentiment_std  positive_mean  negative_mean  pos_neg_ratio sentiment_category
0  2017-09-21           22       5829.0          5864.0                  11693.0          0.1059                   0.0564         0.2492         0.0496         0.0120           4.10            Neutral
1  2017-09-22           18       4236.0          3329.0                   7565.0          0.1124                   0.0598         0.3972         0.0899         0.0276           3.25            Neutral
2  2017-09-23           19       3625.0          3582.0                   7207.0          0.2560                   0.3072         0.3555         0.1259         0.0104       

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os

print("="*70)
print("COMBINED CRYPTOCURRENCY SENTIMENT ANALYSIS (NEWS + TWEETS)")
print("="*70)

# ============================================================
# 1. LOAD DATA
# ============================================================
print("\n📊 Loading sentiment data...")

# Load BTC data
btc_news = pd.read_csv('news_sentiment_bitcoin_daily.csv')
btc_tweets = pd.read_csv('btc_daily_sentiment_2017_2022.csv')

# Load ETH data
eth_news = pd.read_csv('news_sentiment_ethereum_daily.csv')
eth_tweets = pd.read_csv('eth_daily_sentiment_2017_2022.csv')

# Convert dates
btc_news['date'] = pd.to_datetime(btc_news['date'])
btc_tweets['date'] = pd.to_datetime(btc_tweets['date'])
eth_news['date'] = pd.to_datetime(eth_news['date'])
eth_tweets['date'] = pd.to_datetime(eth_tweets['date'])

print(f"\n📅 BTC News: {len(btc_news)} days ({btc_news['date'].min().date()} to {btc_news['date'].max().date()})")
print(f"📅 BTC Tweets: {len(btc_tweets)} days ({btc_tweets['date'].min().date()} to {btc_tweets['date'].max().date()})")
print(f"📅 ETH News: {len(eth_news)} days ({eth_news['date'].min().date()} to {eth_news['date'].max().date()})")
print(f"📅 ETH Tweets: {len(eth_tweets)} days ({eth_tweets['date'].min().date()} to {eth_tweets['date'].max().date()})")

# ============================================================
# 2. DEFINE WEIGHTING SCHEME
# ============================================================
print("\n⚖️ Defining sentiment weighting scheme...")

# Suggested weights based on data characteristics
weighting_scheme = {
    'btc': {
        'news_weight': 0.4,      # News has more authority but less volume
        'tweet_weight': 0.6,      # Tweets have higher volume and engagement
        'rationale': """
            BTC Tweet Weight (0.6):
            - Higher daily volume (avg 3,000+ tweets/day)
            - Engagement metrics available (likes, retweets)
            - Captures retail sentiment effectively
            
            BTC News Weight (0.4):
            - Lower volume but higher authority
            - Professional analysis and institutional perspective
            - Better for detecting major market events
        """
    },
    'eth': {
        'news_weight': 0.35,      # ETH has slightly less news coverage
        'tweet_weight': 0.65,      # ETH has strong community presence
        'rationale': """
            ETH Tweet Weight (0.65):
            - Very active crypto community
            - Developer updates and ecosystem news
            - Strong retail following
            
            ETH News Weight (0.35):
            - Growing but still less than BTC news coverage
            - Technical developments well covered
            - DeFi and NFT news often ETH-focused
        """
    }
}

print("\n📋 BTC Weighting Rationale:")
print(weighting_scheme['btc']['rationale'])
print("\n📋 ETH Weighting Rationale:")
print(weighting_scheme['eth']['rationale'])

# ============================================================
# 3. COMBINE BTC SENTIMENT
# ============================================================
print("\n🪙 Combining BTC sentiment...")

# Merge BTC data
btc_combined = pd.merge(
    btc_news[['date', 'weighted_sentiment_mean', 'news_count']].rename(
        columns={'weighted_sentiment_mean': 'news_sentiment', 'news_count': 'news_volume'}),
    btc_tweets[['date', 'weighted_sentiment_mean', 'tweet_count', 'total_engagement_weight']].rename(
        columns={'weighted_sentiment_mean': 'tweet_sentiment', 'tweet_count': 'tweet_volume'}),
    on='date',
    how='outer'
).sort_values('date')

# Fill missing values (forward fill then backward fill)
btc_combined = btc_combined.fillna(method='ffill').fillna(method='bfill')

# Calculate combined sentiment with weights
btc_combined['combined_sentiment'] = (
    weighting_scheme['btc']['news_weight'] * btc_combined['news_sentiment'] +
    weighting_scheme['btc']['tweet_weight'] * btc_combined['tweet_sentiment']
).round(4)

# Calculate confidence based on data availability
btc_combined['confidence'] = (
    weighting_scheme['btc']['news_weight'] * (btc_combined['news_volume'].notna() * 0.5 + 0.5) +
    weighting_scheme['btc']['tweet_weight'] * (btc_combined['tweet_volume'].notna() * 0.5 + 0.5)
).round(2)

# Determine sentiment category
btc_combined['sentiment_category'] = pd.cut(
    btc_combined['combined_sentiment'],
    bins=[-1, -0.3, -0.1, 0.1, 0.3, 1],
    labels=['Very Negative', 'Negative', 'Neutral', 'Positive', 'Very Positive']
)

print(f"✅ Combined {len(btc_combined)} days for BTC")
print(f"   Date range: {btc_combined['date'].min().date()} to {btc_combined['date'].max().date()}")

# ============================================================
# 4. COMBINE ETH SENTIMENT
# ============================================================
print("\n🪙 Combining ETH sentiment...")

# Merge ETH data
eth_combined = pd.merge(
    eth_news[['date', 'weighted_sentiment_mean', 'news_count']].rename(
        columns={'weighted_sentiment_mean': 'news_sentiment', 'news_count': 'news_volume'}),
    eth_tweets[['date', 'weighted_sentiment_mean', 'tweet_count', 'total_engagement_weight']].rename(
        columns={'weighted_sentiment_mean': 'tweet_sentiment', 'tweet_count': 'tweet_volume'}),
    on='date',
    how='outer'
).sort_values('date')

# Fill missing values
eth_combined = eth_combined.fillna(method='ffill').fillna(method='bfill')

# Calculate combined sentiment with weights
eth_combined['combined_sentiment'] = (
    weighting_scheme['eth']['news_weight'] * eth_combined['news_sentiment'] +
    weighting_scheme['eth']['tweet_weight'] * eth_combined['tweet_sentiment']
).round(4)

# Calculate confidence
eth_combined['confidence'] = (
    weighting_scheme['eth']['news_weight'] * (eth_combined['news_volume'].notna() * 0.5 + 0.5) +
    weighting_scheme['eth']['tweet_weight'] * (eth_combined['tweet_volume'].notna() * 0.5 + 0.5)
).round(2)

# Determine sentiment category
eth_combined['sentiment_category'] = pd.cut(
    eth_combined['combined_sentiment'],
    bins=[-1, -0.3, -0.1, 0.1, 0.3, 1],
    labels=['Very Negative', 'Negative', 'Neutral', 'Positive', 'Very Positive']
)

print(f"✅ Combined {len(eth_combined)} days for ETH")
print(f"   Date range: {eth_combined['date'].min().date()} to {eth_combined['date'].max().date()}")

# ============================================================
# 5. SAVE COMBINED RESULTS
# ============================================================
print("\n💾 Saving combined results...")

# Save BTC combined
btc_combined.to_csv('btc_combined_sentiment.csv', index=False)
print(f"✅ Saved BTC combined to: btc_combined_sentiment.csv")

# Save ETH combined
eth_combined.to_csv('eth_combined_sentiment.csv', index=False)
print(f"✅ Saved ETH combined to: eth_combined_sentiment.csv")

# ============================================================
# 6. STATISTICAL ANALYSIS
# ============================================================
print("\n📊 Statistical Analysis")
print("-" * 50)

# BTC Statistics
print("\n🪙 BTC Combined Sentiment Statistics:")
print(f"   Mean: {btc_combined['combined_sentiment'].mean():.4f}")
print(f"   Median: {btc_combined['combined_sentiment'].median():.4f}")
print(f"   Std Dev: {btc_combined['combined_sentiment'].std():.4f}")
print(f"   Min: {btc_combined['combined_sentiment'].min():.4f}")
print(f"   Max: {btc_combined['combined_sentiment'].max():.4f}")
print(f"   Positive Days: {(btc_combined['combined_sentiment'] > 0).sum()} ({((btc_combined['combined_sentiment'] > 0).sum()/len(btc_combined)*100):.1f}%)")
print(f"   Negative Days: {(btc_combined['combined_sentiment'] < 0).sum()} ({((btc_combined['combined_sentiment'] < 0).sum()/len(btc_combined)*100):.1f}%)")

# ETH Statistics
print("\n🪙 ETH Combined Sentiment Statistics:")
print(f"   Mean: {eth_combined['combined_sentiment'].mean():.4f}")
print(f"   Median: {eth_combined['combined_sentiment'].median():.4f}")
print(f"   Std Dev: {eth_combined['combined_sentiment'].std():.4f}")
print(f"   Min: {eth_combined['combined_sentiment'].min():.4f}")
print(f"   Max: {eth_combined['combined_sentiment'].max():.4f}")
print(f"   Positive Days: {(eth_combined['combined_sentiment'] > 0).sum()} ({((eth_combined['combined_sentiment'] > 0).sum()/len(eth_combined)*100):.1f}%)")
print(f"   Negative Days: {(eth_combined['combined_sentiment'] < 0).sum()} ({((eth_combined['combined_sentiment'] < 0).sum()/len(eth_combined)*100):.1f}%)")

# ============================================================
# 7. CORRELATION ANALYSIS
# ============================================================
print("\n📈 Correlation Analysis")
print("-" * 50)

# Merge BTC and ETH for correlation
combined_correlation = pd.merge(
    btc_combined[['date', 'combined_sentiment']].rename(columns={'combined_sentiment': 'btc_sentiment'}),
    eth_combined[['date', 'combined_sentiment']].rename(columns={'combined_sentiment': 'eth_sentiment'}),
    on='date',
    how='inner'
)

if len(combined_correlation) > 0:
    correlation = combined_correlation['btc_sentiment'].corr(combined_correlation['eth_sentiment'])
    print(f"\n📊 BTC-ETH Combined Sentiment Correlation: {correlation:.4f}")
    
    if correlation > 0.7:
        print("   → Strong positive correlation - sentiments move together")
    elif correlation > 0.3:
        print("   → Moderate positive correlation")
    elif correlation > -0.3:
        print("   → Weak correlation - relatively independent")
    else:
        print("   → Negative correlation - move in opposite directions")

# ============================================================
# 8. ALTERNATIVE WEIGHTING METHODS
# ============================================================
print("\n🎯 Alternative Weighting Methods")
print("-" * 50)

alternative_weights = {
    'volume_based': {
        'description': 'Weight based on daily volume',
        'btc_news_weight': 0.3,
        'btc_tweet_weight': 0.7,
        'eth_news_weight': 0.25,
        'eth_tweet_weight': 0.75,
        'rationale': 'Tweets have much higher daily volume'
    },
    'confidence_based': {
        'description': 'Weight based on confidence scores',
        'btc_news_weight': 0.45,
        'btc_tweet_weight': 0.55,
        'eth_news_weight': 0.4,
        'eth_tweet_weight': 0.6,
        'rationale': 'News has higher confidence per article, but tweets have more data points'
    },
    'equal_weight': {
        'description': 'Equal weight to both sources',
        'btc_news_weight': 0.5,
        'btc_tweet_weight': 0.5,
        'eth_news_weight': 0.5,
        'eth_tweet_weight': 0.5,
        'rationale': 'Simple averaging'
    },
    'dynamic_weight': {
        'description': 'Weight changes based on market conditions',
        'btc_news_weight': '0.3-0.5',
        'btc_tweet_weight': '0.5-0.7',
        'eth_news_weight': '0.25-0.45',
        'eth_tweet_weight': '0.55-0.75',
        'rationale': 'News weight increases during major events, tweets dominate normal days'
    }
}

print("\nAlternative weighting strategies you could try:")
for method, details in alternative_weights.items():
    print(f"\n   {method.replace('_', ' ').title()}:")
    print(f"   • {details['description']}")
    print(f"   • BTC: News {details['btc_news_weight']}, Tweets {details['btc_tweet_weight']}")
    print(f"   • ETH: News {details['eth_news_weight']}, Tweets {details['eth_tweet_weight']}")
    print(f"   • {details['rationale']}")

# ============================================================
# 9. RECOMMENDED WEIGHTS
# ============================================================
print("\n✅ Recommended Weights")
print("-" * 50)

recommendations = """
Based on the data characteristics, I recommend:

BITCOIN (BTC):
📰 News Weight: 0.40
   - 6-7 news articles per day on average
   - Professional analysis, higher authority
   - Better for trend detection

🐦 Tweet Weight: 0.60
   - 3,000+ tweets per day
   - Engagement metrics available
   - Captures retail sentiment effectively

ETHEREUM (ETH):
📰 News Weight: 0.35
   - Slightly less news coverage than BTC
   - Technical developments well covered
   - DeFi/NFT news often ETH-focused

🐦 Tweet Weight: 0.65
   - Very active developer community
   - Strong retail following
   - Higher social media presence

Justification:
1. Tweets provide higher daily volume and granularity
2. News provides higher signal-to-noise ratio
3. ETH has stronger community presence online
4. BTC has more traditional media coverage
5. Combined approach balances retail and institutional sentiment
"""

print(recommendations)

print("\n" + "="*70)
print("✅ COMBINED ANALYSIS COMPLETE")
print("="*70)

COMBINED CRYPTOCURRENCY SENTIMENT ANALYSIS (NEWS + TWEETS)

📊 Loading sentiment data...

📅 BTC News: 2919 days (2017-08-17 to 2025-08-27)
📅 BTC Tweets: 1258 days (2017-09-21 to 2022-07-27)
📅 ETH News: 2870 days (2017-08-17 to 2025-08-27)
📅 ETH Tweets: 1257 days (2017-09-21 to 2021-02-28)

⚖️ Defining sentiment weighting scheme...

📋 BTC Weighting Rationale:

            BTC Tweet Weight (0.6):
            - Higher daily volume (avg 3,000+ tweets/day)
            - Engagement metrics available (likes, retweets)
            - Captures retail sentiment effectively
            
            BTC News Weight (0.4):
            - Lower volume but higher authority
            - Professional analysis and institutional perspective
            - Better for detecting major market events
        

📋 ETH Weighting Rationale:

            ETH Tweet Weight (0.65):
            - Very active crypto community
            - Developer updates and ecosystem news
            - Strong retail following
        

/var/folders/jt/p14c01nx3_g5w98m83g9d33h0000gn/T/ipykernel_29807/3745651197.py:96: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  btc_combined = btc_combined.fillna(method='ffill').fillna(method='bfill')
/var/folders/jt/p14c01nx3_g5w98m83g9d33h0000gn/T/ipykernel_29807/3745651197.py:136: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  eth_combined = eth_combined.fillna(method='ffill').fillna(method='bfill')
